# 18 · Gray-Box AIA72 Complete Multimodal Calibration Diagnostics

Completes Prof. Rami's requested calibration reporting for the frozen 72 h branches: Brier score, **Brier Skill Score against the frozen training climatology**, log loss, ECE, calibration slope/intercept, and reliability diagrams. No recalibration, threshold tuning, or policy redesign occurs here.

The frozen training climatology is inherited from the executed SHARP training/calibration pipeline and is not re-estimated from Cycle 25 or 2026.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

ROOT=Path('/home/abmoses2000')
MASTER=ROOT/'graybox_aia72_master_predictions_v2_20261005'/'graybox_aia72_master_predictions.csv.gz'
MASTER_SHA='17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46'
SHARP=ROOT/'sharp72_predictions_frozen_20261002.csv.gz'
SHARP_SHA='7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b'
OUT=ROOT/'graybox_aia72_calibration_diagnostics_v2_20261006'
if OUT.exists(): raise SystemExit(f'Refusing to overwrite existing output: {OUT}')
(OUT/'figures').mkdir(parents=True)

# Frozen training prevalence from results/sharp72_training_20261002/summary.json
TRAIN_CLIMATOLOGY=0.0576877980145392
SEEDS=[17,29,43]

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
def require(c,m):
    if not c: raise ValueError(m)
require(MASTER.is_file() and SHARP.is_file(),'Missing frozen predictions')
require(sha256(MASTER)==MASTER_SHA,'Master SHA changed')
require(sha256(SHARP)==SHARP_SHA,'SHARP SHA changed')

a=pd.read_csv(MASTER,low_memory=False)
s=pd.read_csv(SHARP,low_memory=False)
a['forecast_case_id']=a['forecast_case_id'].astype(str); s['forecast_case_id']=s['forecast_case_id'].astype(str)
m=a.merge(s,on='forecast_case_id',how='inner',suffixes=('_aia','_sharp'),validate='one_to_one')
require(len(m)==len(a),'Alignment lost rows')
label_a='label_aia' if 'label_aia' in m else 'label'
label_s='label_sharp' if 'label_sharp' in m else label_a
m['label_final']=pd.to_numeric(m[label_a],errors='raise').astype(int)
require(np.array_equal(m['label_final'].to_numpy(),pd.to_numeric(m[label_s],errors='raise').astype(int).to_numpy()),'Labels disagree')

# Robust issue-time recovery, matching Notebook 17.
cands=[c for c in ['issue_utc_aia','issue_utc_sharp','issue_utc','T_REC_dt_aia','T_REC_dt_sharp','T_REC_dt','T_REC_aia','T_REC_sharp','T_REC'] if c in m.columns]
issue=pd.Series(pd.NaT,index=m.index,dtype='datetime64[ns, UTC]')
for c in cands:
    z=pd.to_datetime(m[c],utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
idt=m['forecast_case_id'].str.extract(r':(\d{8}_\d{4})_',expand=False)
z=pd.to_datetime(idt,format='%Y%m%d_%H%M',utc=True,errors='coerce'); take=issue.isna()&z.notna(); issue.loc[take]=z.loc[take]
require(issue.notna().all(),'Unresolved issue times')
m['issue_dt']=issue; m['year']=issue.dt.year.astype(int)
role_col='role_aia' if 'role_aia' in m else 'role'
m['role_final']=m[role_col].astype(str)

m['sharp_probability']=pd.to_numeric(m['probability_gru_mean'],errors='coerce')
m['aia_probability']=pd.to_numeric(m['probability'],errors='coerce')
m['fusion_probability']=0.5*m['sharp_probability']+0.5*m['aia_probability']

def regime(r):
    if r.role_final=='retrospective_cycle25': return 'cycle25_2021_2025_diagnostic'
    if r.role_final=='supplementary_2026': return 'supplementary_2026_locked'
    if r.year<=2019: return 'cycle24_earlier_development'
    return 'other_or_gap'
m['regime']=m.apply(regime,axis=1)

def reliability(y,p,bins=10):
    y=np.asarray(y,int); p=np.asarray(p,float)
    edges=np.linspace(0,1,bins+1); ids=np.minimum(np.digitize(p,edges[1:-1]),bins-1)
    rows=[]
    for b in range(bins):
        sel=ids==b
        rows.append({'bin':b,'left':edges[b],'right':edges[b+1],'count':int(sel.sum()),
                     'mean_probability':float(p[sel].mean()) if sel.any() else np.nan,
                     'observed_fraction':float(y[sel].mean()) if sel.any() else np.nan})
    return pd.DataFrame(rows)

def diagnostics(y,p):
    y=np.asarray(y,int); p=np.asarray(p,float); finite=np.isfinite(p); y=y[finite]; p=p[finite]
    q=np.clip(p,1e-6,1-1e-6); x=np.log(q/(1-q)).reshape(-1,1)
    slope=intercept=np.nan
    if len(y)>=10 and len(np.unique(y))==2:
        lr=LogisticRegression(C=1e6,solver='lbfgs',max_iter=5000).fit(x,y)
        slope=float(lr.coef_[0,0]); intercept=float(lr.intercept_[0])
    rel=reliability(y,p,10)
    ece=float((rel['count']*(rel['mean_probability']-rel['observed_fraction']).abs()).sum()/len(y))
    bs=float(np.mean((p-y)**2))
    bs_ref=float(np.mean((TRAIN_CLIMATOLOGY-y)**2))
    bss=1-bs/bs_ref if bs_ref>0 else np.nan
    ll=float(-np.mean(y*np.log(np.clip(p,1e-12,1-1e-12))+(1-y)*np.log1p(-np.clip(p,1e-12,1-1e-12))))
    return {'cases':len(y),'positives':int(y.sum()),'prevalence':float(y.mean()),'brier':bs,
            'brier_reference_train_climatology':bs_ref,'brier_skill_train_climatology':bss,
            'log_loss':ll,'ece_equal_width_10':ece,'calibration_slope':slope,'calibration_intercept':intercept},rel

summary=[]; rel_all=[]
group_specs=[('regime','regime'),('year','year'),('role','role_final')]
for grouping,col in group_specs:
    for value,g in m.groupby(col,sort=True):
        for branch,pcol in [('sharp','sharp_probability'),('aia','aia_probability'),('sharp_aia','fusion_probability')]:
            y=g['label_final'].to_numpy(int); p=g[pcol].to_numpy(float); finite=np.isfinite(p)
            d,rel=diagnostics(y[finite],p[finite])
            summary.append({'grouping':grouping,'group':str(value),'branch':branch,**d})
            rel.insert(0,'branch',branch); rel.insert(0,'group',str(value)); rel.insert(0,'grouping',grouping)
            rel_all.append(rel)
summary=pd.DataFrame(summary); reliability_bins=pd.concat(rel_all,ignore_index=True)
summary.to_csv(OUT/'calibration_diagnostics.csv',index=False)
reliability_bins.to_csv(OUT/'reliability_bins.csv',index=False)

order=['cycle24_earlier_development','cycle25_2021_2025_diagnostic','supplementary_2026_locked']
reg=summary[(summary.grouping=='regime') & summary.group.isin(order)].copy()
reg['group']=pd.Categorical(reg['group'],categories=order,ordered=True); reg=reg.sort_values(['group','branch'])

# One reliability plot per regime to keep curves readable.
for rg in order:
    fig,ax=plt.subplots(figsize=(6.5,6))
    ax.plot([0,1],[0,1],linestyle='--',linewidth=1,label='ideal')
    for branch in ['sharp','aia','sharp_aia']:
        z=reliability_bins[(reliability_bins.grouping=='regime')&(reliability_bins.group==rg)&(reliability_bins.branch==branch)&(reliability_bins['count']>0)]
        ax.plot(z.mean_probability,z.observed_fraction,marker='o',label=branch)
    ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed flare fraction'); ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_title(f'Reliability diagram — {rg}'); ax.legend(); plt.tight_layout(); plt.savefig(OUT/'figures'/f'reliability_{rg}.png',dpi=220); plt.close()

for metric,title,fname in [
    ('brier_skill_train_climatology','Brier Skill Score vs frozen training climatology','bss_by_regime.png'),
    ('ece_equal_width_10','Expected calibration error','ece_by_regime.png'),
    ('calibration_slope','Calibration slope','slope_by_regime.png'),
    ('calibration_intercept','Calibration intercept','intercept_by_regime.png')]:
    piv=reg.pivot(index='group',columns='branch',values=metric)
    ax=piv.plot(kind='bar',figsize=(10,5)); ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel(metric)
    if metric=='calibration_slope': ax.axhline(1.0,linestyle='--',linewidth=1)
    if metric=='calibration_intercept': ax.axhline(0.0,linestyle='--',linewidth=1)
    plt.xticks(rotation=15,ha='right'); plt.tight_layout(); plt.savefig(OUT/'figures'/fname,dpi=220); plt.close()

protocol={
 'status':'POSTHOC_MULTIMODAL_CALIBRATION_DIAGNOSTICS_COMPLETE',
 'created_utc':datetime.now(timezone.utc).isoformat(),'horizon_hours':72,
 'new_model_fitting':False,'new_threshold_tuning':False,'new_policy_selection':False,'new_policy_redesign':False,
 'train_climatology':TRAIN_CLIMATOLOGY,
 'brier_skill_reference':'frozen training prevalence from executed SHARP72 training summary; no later-period prevalence refit',
 'evidence_boundary':{'cycle24':'development/descriptive','cycle25':'diagnostic/post-hoc for v3','2026':'already-spent clean evaluation; descriptive only here'},
 'source_sha256':{'master_predictions':MASTER_SHA,'sharp_predictions':SHARP_SHA},
 'outputs':['calibration_diagnostics.csv','reliability_bins.csv','figures/']}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')

print('===== COMPLETE MULTIMODAL CALIBRATION DIAGNOSTICS =====')
print('Frozen training climatology:',TRAIN_CLIMATOLOGY)
print(reg[['group','branch','cases','positives','prevalence','brier','brier_skill_train_climatology','log_loss','ece_equal_width_10','calibration_slope','calibration_intercept']].round(6).to_string(index=False))
print('\nOutputs:',OUT)


## Interpretation guardrails

- Brier Skill Score uses the **frozen training climatology**, not the realized prevalence of each later regime.
- Reliability diagrams are descriptive on already-seen/spent periods and do not authorize recalibration.
- Slope = 1 and intercept = 0 are ideal descriptive calibration targets; departures are evidence of miscalibration/shift, not a licence to refit on the same evaluation period.
- These outputs complete calibration reporting; they do not replace the earlier raw/Platt/isotonic method-selection experiment.
